# License Plate Detection — Evaluation & Comparison
**CMPS 261 — Machine Learning Project**

Loads saved metrics from both models, compares performance, and generates final comparison plots.

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from PIL import Image
import random

RESULTS_DIR = '../results'

## 1. Load Metrics from Both Models

In [ ]:
with open(os.path.join(RESULTS_DIR, 'yolo_metrics.json')) as f:
    yolo = json.load(f)

with open(os.path.join(RESULTS_DIR, 'fasterrcnn_metrics.json')) as f:
    frcnn = json.load(f)

print('YOLOv8s metrics:')
print(json.dumps(yolo, indent=2))
print()
print('Faster R-CNN metrics:')
print(json.dumps(frcnn, indent=2))

## 2. Side-by-Side Metric Comparison Table

In [ ]:
rows = [
    {
        'Model'    : 'YOLOv8s',
        'Precision': yolo['precision'],
        'Recall'   : yolo['recall'],
        'mAP@0.5'  : yolo['map50'],
        'mAP@0.5:0.95': yolo['map50_95'],
        'F1'       : round(2 * yolo['precision'] * yolo['recall'] / (yolo['precision'] + yolo['recall']), 4),
    },
    {
        'Model'    : 'Faster R-CNN',
        'Precision': frcnn['precision'],
        'Recall'   : frcnn['recall'],
        'mAP@0.5'  : '-',
        'mAP@0.5:0.95': '-',
        'F1'       : frcnn['f1'],
    },
]

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
df

## 3. Bar Chart Comparison

In [ ]:
# Metrics available for both models
labels = ['Precision', 'Recall', 'F1']
yolo_vals  = [yolo['precision'], yolo['recall'],
               round(2 * yolo['precision'] * yolo['recall'] / (yolo['precision'] + yolo['recall']), 4)]
frcnn_vals = [frcnn['precision'], frcnn['recall'], frcnn['f1']]

x     = np.arange(len(labels))
width = 0.35
colors = ['#4C9BE8', '#E87B4C']

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, yolo_vals,  width, label='YOLOv8s',      color=colors[0], alpha=0.85)
bars2 = ax.bar(x + width/2, frcnn_vals, width, label='Faster R-CNN', color=colors[1], alpha=0.85)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — License Plate Detection')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=150)
plt.show()
print('Saved: results/model_comparison.png')

## 4. Side-by-Side Predictions (YOLOv8s vs Faster R-CNN)

In [ ]:
import torch
import sys
sys.path.append('..')
from ultralytics import YOLO as UltralyticsYOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms.functional as TF

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Load YOLOv8s
yolo_model = UltralyticsYOLO('../models/yolov8s_best.pt')

# Load Faster R-CNN
def load_frcnn(path, device):
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    m = fasterrcnn_resnet50_fpn_v2(weights=weights)
    in_features = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    m.load_state_dict(torch.load(path, map_location=device))
    m.to(device).eval()
    return m

frcnn_model = load_frcnn('../models/fasterrcnn_best.pth', DEVICE)
print('Both models loaded.')

In [ ]:
test_img_dir = '../data/yolo/images/test'
test_images  = random.sample(os.listdir(test_img_dir), 4)

fig, axes = plt.subplots(4, 3, figsize=(14, 16))
CONF = 0.5

for row_idx, fname in enumerate(test_images):
    img_path = os.path.join(test_img_dir, fname)
    img_pil  = Image.open(img_path).convert('RGB')
    img_np   = np.array(img_pil)

    # Col 0: original
    axes[row_idx][0].imshow(img_np)
    axes[row_idx][0].set_title('Original', fontsize=9)
    axes[row_idx][0].axis('off')

    # Col 1: YOLOv8s
    yolo_result = yolo_model.predict(img_path, conf=CONF, verbose=False)[0]
    axes[row_idx][1].imshow(img_np)
    for box in yolo_result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = box.conf[0].item()
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='lime', facecolor='none')
        axes[row_idx][1].add_patch(rect)
        axes[row_idx][1].text(x1, y1-4, f'{conf:.2f}', color='lime', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][1].set_title('YOLOv8s', fontsize=9)
    axes[row_idx][1].axis('off')

    # Col 2: Faster R-CNN
    img_tensor = TF.to_tensor(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = frcnn_model(img_tensor)[0]
    axes[row_idx][2].imshow(img_np)
    for box, score in zip(pred['boxes'], pred['scores']):
        if score < CONF: continue
        x1, y1, x2, y2 = box.cpu().tolist()
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='#FF6B6B', facecolor='none')
        axes[row_idx][2].add_patch(rect)
        axes[row_idx][2].text(x1, y1-4, f'{score:.2f}', color='#FF6B6B', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][2].set_title('Faster R-CNN', fontsize=9)
    axes[row_idx][2].axis('off')

plt.suptitle('Side-by-Side: Original | YOLOv8s | Faster R-CNN', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'side_by_side_comparison.png'), dpi=150)
plt.show()
print('Saved: results/side_by_side_comparison.png')

## 5. Analysis & Discussion

### Results Summary

| Model | Precision | Recall | F1 | mAP@0.5 |
|-------|-----------|--------|----|---------|
| YOLOv8s | 0.9182 | 0.9155 | 0.9168 | 0.9467 |
| Faster R-CNN | 0.8732 | 0.8732 | 0.8732 | — |

**YOLOv8s outperforms Faster R-CNN on this dataset.**

### Why YOLOv8s wins on this dataset

**YOLOv8s (single-stage detector)**
- Predicts bounding boxes and classes in a single forward pass — very fast
- Has heavy built-in augmentation (mosaic, mixup, HSV shifts) that effectively expands small datasets
- Anchor-free detection with a feature pyramid neck for multi-scale detection
- Pretrained on COCO (80 classes), fine-tuned on our 433-image dataset
- With only 433 images, the built-in augmentation is the key advantage

**Faster R-CNN (two-stage detector)**
- Stage 1: Region Proposal Network (RPN) generates candidate regions
- Stage 2: Classifies and refines each region independently
- 43M parameters — large model prone to overfitting on small datasets
- No built-in augmentation; we added manual flips and color jitter but it wasn't enough
- Early stopping kicked in at epoch 13 to prevent further overfitting

### Conclusion

For license plate detection with limited data, **YOLOv8s is the better choice** — faster, more accurate, and more robust to small dataset sizes due to its built-in augmentation pipeline.

For real-time deployment (dashcam, traffic camera) → **YOLOv8s** is strongly preferred.  
Faster R-CNN would require significantly more training data to close the gap.